# Partial Credit and Generalized Partial Credit Models with Latent Regression

**Reference:** Furr, D. C. (2017). *Partial credit and generalized partial credit models with latent regression.* Stan case study, March 6, 2017.

This notebook replicates Furr (2017) using **Python + Stan** via **cmdstanpy**.

**Workflow:**
1. Simulate data (numpy / scipy) — same parameters as the paper's R code
2. Write readable `.stan` model files to disk
3. Compile and sample with **cmdstanpy** (CmdStan backend)
4. Diagnose and assess parameter recovery

**Installation:**
```bash
pip install cmdstanpy matplotlib numpy scipy pandas
python -c "import cmdstanpy; cmdstanpy.install_cmdstan()"
```

---
## Data Sources — Clarification

| | What it provides | Used in this notebook? |
|---|---|---|
| **edstan** (R package) | Stan `.stan` model files + helper functions (`irt_data`, `irt_stan`) | Stan code only (not data) |
| **TAM** (R package) | TIMSS 2011 maths data (`data.timssAusTwn.scored`) | Section 3 of paper only — **not simulated here** |
| **Simulation** (Sections 1.4 & 2.3) | Synthetic data generated from known parameters | **Yes — this is what this notebook replicates** |

**All data in this notebook is generated synthetically** using the same parameters as the R code in Sections 1.4 and 2.3 of the paper. No package data is used. The edstan package contributes the Stan model code (shown below); the TAM package contributes the real-data example in Section 3, which this notebook does not cover.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.special import softmax
import cmdstanpy
from cmdstanpy import CmdStanModel
import os

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

print(f'cmdstanpy version: {cmdstanpy.__version__}')

## Python Helper Functions

These translate the R/Stan helper functions into Python:
- `pcm_probs` / `gpcm_probs` — compute category probabilities (used for simulation)
- `obtain_adjustments` — replicates edstan's covariate scaling (identical logic)

> The `lambda` back-transform is handled inside Stan's `generated quantities` block — no Python post-processing needed.

In [ ]:
def pcm_probs(theta, beta):
    """PCM response probabilities for one person-item pair."""
    unsummed = np.concatenate([[0.0], theta - np.asarray(beta)])
    return softmax(np.cumsum(unsummed))


def gpcm_probs(theta, alpha, beta):
    """GPCM response probabilities for one person-item pair."""
    unsummed = np.concatenate([[0.0], alpha * theta - np.asarray(beta)])
    return softmax(np.cumsum(unsummed))


def simulate_response(probs):
    """Draw a categorical response from a probability vector."""
    return np.random.choice(len(probs), p=probs)


def obtain_adjustments(W):
    """
    Compute centering/scaling adjustments for person covariates.
    Exactly replicates edstan's obtain_adjustments() Stan function.

    Returns adj of shape (2, K):
      adj[0, k] = shift value (subtract from column k)
      adj[1, k] = scale value (divide by after shifting)

    Rules:
      Intercept (k=0): shift=0, scale=1  → unchanged
      Binary column  : shift=mean, scale=max-min
      Continuous col : shift=mean, scale=2*std
    """
    K = W.shape[1]
    adj = np.zeros((2, K))
    adj[0, 0] = 0.0
    adj[1, 0] = 1.0
    for k in range(1, K):
        col = W[:, k]
        n_unique = len(np.unique(col))
        adj[0, k] = col.mean()
        adj[1, k] = (col.max() - col.min()) if n_unique == 2 else (col.std() * 2.0)
    return adj

---
## Stan Code: Original vs. Readable Rewrite

This section shows the Stan code from the paper (edstan package source), then a cleaner
annotated version that is mathematically identical but easier to read.

> **Q2 answer:** The Stan code shown here *is* the original source — it is printed verbatim from
> the edstan package files (`extdata/pcm_simple.stan`, `extdata/pcm_latent_reg.stan`,
> `extdata/gpcm_latent_reg.stan`). The paper prints these directly using `cat(readLines(...))`.  
> **Q3 answer:** The "Readable rewrite" below is a semantically equivalent but more
> commented and clearly structured version — same numerical results, easier to understand.

### Simple PCM — Original Stan Code (edstan / Section 1.2)

```stan
functions {
  real pcm(int y, real theta, vector beta) {
    vector[rows(beta) + 1] unsummed;
    vector[rows(beta) + 1] probs;
    unsummed = append_row(rep_vector(0.0, 1), theta - beta);
    probs = softmax(cumulative_sum(unsummed));
    return categorical_lpmf(y + 1 | probs);
  }
}
data {
  int<lower=1> I;               // # items
  int<lower=1> J;               // # persons
  int<lower=1> N;               // # responses
  int<lower=1,upper=I> ii[N];   // i for n
  int<lower=1,upper=J> jj[N];   // j for n
  int<lower=0> y[N];            // response for n; y = 0, 1 ... m_i
}
transformed data {
  int m;                        // # parameters per item (same for all items)
  m = max(y);
}
parameters {
  vector[m] beta[I];
  vector[J] theta;
  real<lower=0> sigma;
}
model {
  for(i in 1:I)
    beta[i] ~ normal(0, 3);
  theta ~ normal(0, sigma);
  sigma ~ exponential(.1);
  for (n in 1:N)
    target += pcm(y[n], theta[jj[n]], beta[ii[n]]);
}
```

### Simple PCM — Readable Rewrite

Differences from the original:
- Renamed `pcm()` → `pcm_loglik()` to be explicit it returns a log-probability
- **Replaced `softmax + categorical_lpmf` with `categorical_logit_lpmf`** — the built-in Stan function that accepts unnormalized log-probabilities directly (no manual softmax needed)
- Used modern Stan array syntax `array[N] int` instead of deprecated `int arr[N]`
- Added comments explaining each block and the cumulative-sum identity

```stan
// ================================================================
// Simple Partial Credit Model (PCM) — Masters (1982)
// Intercept-only ability distribution (no latent regression).
// All items must have the same number of response categories.
//
// Model:
//   Y_ij in {0, 1, ..., m}  (ordered polytomous response)
//   P(Y=y | theta, beta) proportional to exp[ sum_{s=1}^{y} (theta - beta_s) ]
//   theta_j ~ Normal(0, sigma)
// ================================================================

functions {

  // Log-probability of response y under the Partial Credit Model.
  //
  // Cumulative-sum trick:
  //   log_num[1]   = 0                          (category 0, reference)
  //   log_num[k+1] = log_num[k] + (theta - beta[k])   for k = 1,...,m
  //
  // These are the UNNORMALIZED log-probabilities (log-odds) for each category.
  // categorical_logit_lpmf applies softmax internally:
  //   categorical_logit_lpmf(k | eta) == log softmax(eta)[k]
  //                                    == eta[k] - log_sum_exp(eta)
  // y+1 because Stan's categorical distribution is 1-indexed.
  real pcm_loglik(int y, real theta, vector beta) {
    int m = rows(beta);
    vector[m + 1] log_num;      // unnormalized log-probs (= log-odds)
    log_num[1] = 0.0;           // category 0: reference (log-odds = 0)
    for (k in 1:m)
      log_num[k + 1] = log_num[k] + (theta - beta[k]);
    return categorical_logit_lpmf(y + 1 | log_num);
  }

}

data {
  int<lower=1> I;                      // number of items
  int<lower=1> J;                      // number of persons
  int<lower=1> N;                      // number of observed responses
  array[N] int<lower=1,upper=I> ii;    // item index for each response
  array[N] int<lower=1,upper=J> jj;    // person index for each response
  array[N] int<lower=0>         y;     // observed response in {0,...,m}
}

transformed data {
  // All items have the same number of steps (m = max observed category).
  int m = max(y);
}

parameters {
  array[I] vector[m] beta;   // beta[i] = m step difficulties for item i
  vector[J]          theta;  // person abilities
  real<lower=0>      sigma;  // ability SD
}

model {
  // --- Priors ---
  sigma ~ exponential(0.1);
  for (i in 1:I)
    beta[i] ~ normal(0, 3);
  theta ~ normal(0, sigma);

  // --- Likelihood ---
  for (n in 1:N)
    target += pcm_loglik(y[n], theta[jj[n]], beta[ii[n]]);
}
```

### Full PCM with Latent Regression — Original Stan Code (edstan / Section 1.3)

```stan
functions {
  real pcm(int y, real theta, vector beta) {
    vector[rows(beta) + 1] unsummed;
    vector[rows(beta) + 1] probs;
    unsummed = append_row(rep_vector(0.0, 1), theta - beta);
    probs = softmax(cumulative_sum(unsummed));
    return categorical_lpmf(y + 1 | probs);
  }
  matrix obtain_adjustments(matrix W) {
    real min_w;
    real max_w;
    int minmax_count;
    matrix[2, cols(W)] adj;
    adj[1, 1] = 0;
    adj[2, 1] = 1;
    if(cols(W) > 1) {
      for(k in 2:cols(W)) {
        min_w = min(W[1:rows(W), k]);
        max_w = max(W[1:rows(W), k]);
        minmax_count = 0;
        for(j in 1:rows(W))
          minmax_count = minmax_count + W[j,k] == min_w || W[j,k] == max_w;
        if(minmax_count == rows(W)) {
          adj[1, k] = mean(W[1:rows(W), k]);
          adj[2, k] = (max_w - min_w);
        } else {
          adj[1, k] = mean(W[1:rows(W), k]);
          adj[2, k] = sd(W[1:rows(W), k]) * 2;
        }
      }
    }
    return adj;
  }
}
data {
  int<lower=1> I;               // # items
  int<lower=1> J;               // # persons
  int<lower=1> N;               // # responses
  int<lower=1,upper=I> ii[N];   // i for n
  int<lower=1,upper=J> jj[N];   // j for n
  int<lower=0> y[N];            // response for n; y = 0, 1 ... m_i
  int<lower=1> K;               // # person covariates
  matrix[J,K] W;                // person covariate matrix
}
transformed data {
  int m[I];                     // # parameters per item
  int pos[I];                   // first position in beta vector for item
  matrix[2,K] adj;
  matrix[J,K] W_adj;
  m = rep_array(0, I);
  for(n in 1:N)
    if(y[n] > m[ii[n]]) m[ii[n]] = y[n];
  pos[1] = 1;
  for(i in 2:(I))
    pos[i] = m[i-1] + pos[i-1];
  adj = obtain_adjustments(W);
  for(k in 1:K) for(j in 1:J)
      W_adj[j,k] = (W[j,k] - adj[1,k]) / adj[2,k];
}
parameters {
  vector[sum(m)-1] beta_free;
  vector[J] theta;
  real<lower=0> sigma;
  vector[K] lambda_adj;
}
transformed parameters {
  vector[sum(m)] beta;
  beta[1:(sum(m)-1)] = beta_free;
  beta[sum(m)] = -1*sum(beta_free);
}
model {
  target += normal_lpdf(beta | 0, 3);
  theta ~ normal(W_adj*lambda_adj, sigma);
  lambda_adj ~ student_t(3, 0, 1);
  sigma ~ exponential(.1);
  for (n in 1:N)
    target += pcm(y[n], theta[jj[n]],  segment(beta, pos[ii[n]], m[ii[n]]));
}
generated quantities {
  vector[K] lambda;
  lambda[2:K] = lambda_adj[2:K] ./ to_vector(adj[2,2:K]);
  lambda[1] = W_adj[1, 1:K]*lambda_adj[1:K] - W[1, 2:K]*lambda[2:K];
}
```

### Full PCM with Latent Regression — Readable Rewrite

What changed from the original:
- `pcm()` → `pcm_loglik()` returning log-prob via **`categorical_logit_lpmf`** (no manual softmax)
- `obtain_adjustments()` → `get_adjustments()` with clearer logic and comments
- `m[]` (ambiguous) → `steps[]` (number of steps per item)
- `pos[]` (cryptic) → `beta_start[]` (starting position of each item's betas)
- Full doc-comments on every block explaining the statistical meaning
- Modern Stan syntax

```stan
// ================================================================
// Partial Credit Model (PCM) with Latent Regression
// Furr (2017) / edstan — annotated readable version
//
// MODEL:
//   Y_ij in {0,...,m_i}   (ordered polytomous, items may differ)
//   P(Y=y | theta, beta) proportional to exp[ sum_{s=1}^{y} (theta_j - beta_{is}) ]
//   theta_j ~ Normal(w_j' * lambda, sigma^2)   [latent regression]
//
// CONSTRAINT:
//   sum_is(beta_{is}) = 0  — the last beta is fixed as -sum of all others.
//
// COVARIATE SCALING:
//   W is centered and scaled (W_adj) so that lambda_adj ~ t_3(0,1)
//   stays weakly informative. Generated quantities back-transform lambda.
// ================================================================

functions {

  // ------------------------------------------------------------------
  // pcm_loglik: log P(Y = y | theta, beta) under the PCM.
  //
  // Builds unnormalized log-probs (log-odds) via cumulative sum, then
  // calls categorical_logit_lpmf which applies softmax internally.
  //
  //   log_num[1]   = 0
  //   log_num[k+1] = log_num[k] + (theta - beta[k]),  k = 1,...,m
  //
  //   categorical_logit_lpmf(y+1 | log_num)
  //       = log_num[y+1] - log_sum_exp(log_num)   [equivalent formula]
  // ------------------------------------------------------------------
  real pcm_loglik(int y, real theta, vector beta) {
    int m = rows(beta);
    vector[m + 1] log_num;
    log_num[1] = 0.0;
    for (k in 1:m)
      log_num[k + 1] = log_num[k] + (theta - beta[k]);
    return categorical_logit_lpmf(y + 1 | log_num);
  }

  // ------------------------------------------------------------------
  // get_adjustments: centering and scaling for each covariate column.
  //
  // Returns 2 x K matrix:
  //   adj[1, k] = shift (amount to subtract)
  //   adj[2, k] = scale (amount to divide by after shifting)
  //
  // Intercept (k=1): shift=0, scale=1       (unchanged)
  // Binary column  : shift=mean, scale=max-min
  // Continuous col : shift=mean, scale=2*SD
  // ------------------------------------------------------------------
  matrix get_adjustments(matrix W) {
    int J = rows(W);
    int K = cols(W);
    matrix[2, K] adj;
    adj[1, 1] = 0.0;
    adj[2, 1] = 1.0;
    for (k in 2:K) {
      real w_min = min(col(W, k));
      real w_max = max(col(W, k));
      // Count rows at min or max — all at extremes means binary covariate
      int n_at_extremes = 0;
      for (j in 1:J)
        n_at_extremes += (W[j, k] == w_min || W[j, k] == w_max);
      adj[1, k] = mean(col(W, k));
      if (n_at_extremes == J)
        adj[2, k] = w_max - w_min;     // binary
      else
        adj[2, k] = sd(col(W, k)) * 2; // continuous
    }
    return adj;
  }

}

data {
  int<lower=1> I;                     // number of items
  int<lower=1> J;                     // number of persons
  int<lower=1> N;                     // total observed responses
  int<lower=1> K;                     // number of covariates (incl. intercept)
  array[N] int<lower=1,upper=I> ii;
  array[N] int<lower=1,upper=J> jj;
  array[N] int<lower=0>         y;
  matrix[J, K] W;
}

transformed data {
  // steps[i] = max observed category = number of step difficulties for item i
  array[I] int steps = rep_array(0, I);
  for (n in 1:N)
    if (y[n] > steps[ii[n]]) steps[ii[n]] = y[n];

  // beta_start[i] = index in flat beta vector where item i begins
  array[I] int beta_start;
  beta_start[1] = 1;
  for (i in 2:I)
    beta_start[i] = beta_start[i - 1] + steps[i - 1];
  int total_steps = sum(steps);

  // Center and scale covariates
  matrix[2, K] adj    = get_adjustments(W);
  matrix[J, K] W_adj;
  for (k in 1:K)
    W_adj[, k] = (W[, k] - adj[1, k]) / adj[2, k];
}

parameters {
  vector[total_steps - 1] beta_free;  // (total - 1) free; last is constrained
  vector[J]      theta;
  real<lower=0>  sigma;
  vector[K]      lambda_adj;          // regression coefficients, adjusted scale
}

transformed parameters {
  // Sum-to-zero constraint: last beta = -sum of all others
  vector[total_steps] beta;
  beta[1:(total_steps - 1)] = beta_free;
  beta[total_steps]          = -sum(beta_free);
}

model {
  // --- Priors ---
  target += normal_lpdf(beta | 0, 3);
  lambda_adj ~ student_t(3, 0, 1);
  sigma      ~ exponential(0.1);
  theta      ~ normal(W_adj * lambda_adj, sigma);

  // --- Likelihood ---
  for (n in 1:N) {
    int i = ii[n];
    target += pcm_loglik(y[n], theta[jj[n]], segment(beta, beta_start[i], steps[i]));
  }
}

generated quantities {
  // Back-transform lambda to original covariate scale
  vector[K] lambda;
  for (k in 2:K)
    lambda[k] = lambda_adj[k] / adj[2, k];
  lambda[1] = lambda_adj[1] - dot_product(adj[1, 2:K], lambda[2:K]);
}
```

### GPCM with Latent Regression — Readable Rewrite

Only three changes from the PCM readable version:
1. Add `alpha` discrimination parameter (one per item, constrained positive)
2. Replace `theta - beta[k]` with `alpha * theta - beta[k]`
3. Fix `sigma = 1.0` (not estimated); add `alpha ~ lognormal(1, 1)`

```stan
// ================================================================
// Generalized Partial Credit Model (GPCM) with Latent Regression
// Furr (2017) / edstan — annotated readable version
//
// Extends PCM by adding discrimination alpha_i per item.
// When alpha_i = 1 for all i, reduces exactly to the PCM.
//
// MODEL:
//   P(Y=y | theta, alpha, beta) proportional to
//       exp[ sum_{s=1}^{y} (alpha_i * theta_j - beta_{is}) ]
//   theta_j ~ Normal(w_j' * lambda, 1)   [variance FIXED to 1]
//
// IDENTIFICATION:
//   PCM:  alpha_i = 1 fixed,  sigma estimated  → sigma carries the scale
//   GPCM: alpha_i estimated,  Var(theta) = 1   → unit variance fixes the scale
// ================================================================

functions {

  // gpcm_loglik: identical structure to pcm_loglik, but alpha scales theta.
  // categorical_logit_lpmf applies softmax to log_num internally.
  real gpcm_loglik(int y, real theta, real alpha, vector beta) {
    int m = rows(beta);
    vector[m + 1] log_num;
    log_num[1] = 0.0;
    for (k in 1:m)
      log_num[k + 1] = log_num[k] + (alpha * theta - beta[k]);
    return categorical_logit_lpmf(y + 1 | log_num);
  }

  // get_adjustments: identical to the PCM version
  matrix get_adjustments(matrix W) { /* ... same as PCM ... */ }

}

// data block: identical to the PCM version

// transformed data block: identical to the PCM version

parameters {
  // Item discriminations: must be positive
  vector<lower=0>[I] alpha;

  vector[total_steps - 1] beta_free;
  vector[J]      theta;
  // NOTE: sigma is NOT a parameter in GPCM — Var(theta) is fixed to 1.
  vector[K]      lambda_adj;
}

transformed parameters {
  // Identical sum-to-zero constraint as PCM
  vector[total_steps] beta;
  beta[1:(total_steps - 1)] = beta_free;
  beta[total_steps]          = -sum(beta_free);
}

model {
  // --- Priors ---
  // LogNormal(1, 1): positive, weakly informative around moderate discrimination
  alpha      ~ lognormal(1, 1);
  target += normal_lpdf(beta | 0, 3);
  lambda_adj ~ student_t(3, 0, 1);
  theta      ~ normal(W_adj * lambda_adj, 1.0);  // variance FIXED to 1

  // --- Likelihood (GPCM) ---
  for (n in 1:N) {
    int i = ii[n];
    target += gpcm_loglik(
      y[n], theta[jj[n]], alpha[i],
      segment(beta, beta_start[i], steps[i])
    );
  }
}

// generated quantities: identical to PCM version
```

### Stan → cmdstanpy Workflow

**Compiling a `.stan` file:**
```python
from cmdstanpy import CmdStanModel
model = CmdStanModel(stan_file='pcm_latent_reg.stan')
```

**Passing data — Python 0-indexed arrays must be shifted to Stan's 1-indexed:**
```python
stan_data = {
    'I': I, 'J': J, 'N': N, 'K': K,
    'ii': (ii + 1).tolist(),   # item indices: 0-indexed → 1-indexed
    'jj': (jj + 1).tolist(),   # person indices: 0-indexed → 1-indexed
    'y':  y.tolist(),           # responses stay 0-indexed (Stan lower=0)
    'W':  W.tolist(),
}
```

**Sampling:**
```python
fit = model.sample(data=stan_data, chains=4,
                   iter_warmup=500, iter_sampling=500,
                   adapt_delta=0.9, seed=42)
```

**Extracting results:**
```python
fit.stan_variable('lambda')   # shape (draws, K) — back-transformed by generated quantities
fit.stan_variable('beta')     # shape (draws, total_steps)
fit.stan_variable('sigma')    # shape (draws,)
fit.summary()                 # pandas DataFrame: Mean, StdDev, R_hat, ESS_bulk
fit.diagnose()                # text report: divergences, max tree depth, E-BFMI
```

**Why `categorical_logit_lpmf` is the right Stan likelihood:**

$$
\texttt{categorical\_logit\_lpmf}(y{+}1 \mid \eta)
= \log\,\text{softmax}(\eta)_{y+1}
= \eta_{y+1} - \log\!\sum_k e^{\eta_k}
$$

| Stan (original edstan) | Stan (readable rewrite) |
|---|---|
| `softmax(cumulative_sum(unsummed))[y+1]` via `categorical_lpmf` | `categorical_logit_lpmf(y+1 \| log_num)` |

Both are numerically identical — `categorical_logit_lpmf` just skips the explicit `softmax` call.

---
## Section 1: PCM Simulation and Parameter Recovery

### 1.1 Model

$$
\Pr(Y_{ij} = y \mid \theta_j, \boldsymbol{\beta}_i) =
\frac{\exp\sum_{s=1}^{y}(\theta_j - \beta_{is})}
     {1 + \sum_{k=1}^{m_i}\exp\sum_{s=1}^{k}(\theta_j - \beta_{is})}
\qquad
\theta_j \sim \mathcal{N}(\boldsymbol{w}_j'\boldsymbol{\lambda},\; \sigma^2)
$$

**Constraint:** $\sum_{i,s}\beta_{is} = 0$  
**Priors:** $\sigma \sim \text{Exp}(0.1)$,  $\beta_{is} \sim \mathcal{N}(0,9)$,  $\boldsymbol{\lambda} \sim t_3(0,1)$

### 1.2 Simulation Design (matches R code in Furr 2017, Section 1.4)

$I = 20$ items, $m_i = 2$ steps (3 categories), $J = 500$ persons, $K = 4$ covariates:
1. Intercept $w_1 = 1$
2. Continuous $w_2 \sim \mathcal{N}(10, 5)$
3. Binary $w_3 \sim \text{Bernoulli}(0.5)$
4. Interaction $w_4 = w_2 \times w_3$

True parameters: $\sigma = 1.2$, $\boldsymbol{\lambda} = (-0.5,\; 0.05,\; 0.5,\; -0.025)$

In [ ]:
np.random.seed(SEED)

J = 500;  I = 20;  m = 2            # persons, items, steps-per-item
sigma_true  = 1.2
lambda_true = np.array([-0.5, 0.05, 0.5, -0.025])

# Person covariates (matching paper's R code)
w2 = np.random.normal(10, 5, J)
w3 = np.random.binomial(1, 0.5, J)
W  = np.column_stack([np.ones(J), w2, w3, w2 * w3])   # shape (J, K=4)
K  = W.shape[1]

# Item step difficulties (matching paper's R code)
Beta_unc = np.zeros((I, m))
Beta_unc[:, 0] = np.linspace(-1, 0, I)
pattern  = np.tile([0.2, 0.4, 0.6, 0.8], (I // 4) + 1)[:I]
Beta_unc[:, 1] = Beta_unc[:, 0] + pattern
Beta = Beta_unc - Beta_unc.mean()   # sum-to-zero constraint: sum(Beta) == 0

# Long-format observation indices
N  = I * J
ii = np.tile(np.arange(I), J).astype(int)
jj = np.repeat(np.arange(J), I).astype(int)

print(f'J={J}, I={I}, m={m}, N={N}, K={K}')
print(f'True sigma : {sigma_true}')
print(f'True lambda: {lambda_true}')
print(f'Beta  sum  : {Beta.sum():.10f}  (should be 0)')

In [ ]:
# Generate person abilities from the latent regression model
mu_theta  = W @ lambda_true
pcm_theta = np.random.normal(mu_theta, sigma_true, J)

# Generate item responses
pcm_y = np.array([
    simulate_response(pcm_probs(pcm_theta[jj[n]], Beta[ii[n]]))
    for n in range(N)
], dtype=int)

print('Response frequency:')
for cat in range(m + 1):
    print(f'  Category {cat}: {(pcm_y == cat).mean()*100:.1f}%')
print(f'\nAbility: mean={pcm_theta.mean():.3f}, std={pcm_theta.std():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: latent regression structure (replicates Figure in Furr 2017)
ax = axes[0]
x, lam = np.linspace(0, 20, 200), lambda_true
ax.plot(x, lam[0] + lam[1]*x,                      'r-', lw=2, label='$w_3=0$')
ax.plot(x, lam[0] + lam[1]*x + lam[2] + lam[3]*x, 'b-', lw=2, label='$w_3=1$')
ax.set_xlabel('Continuous covariate ($w_2$)')
ax.set_ylabel('Mean ability $E[\\theta \\mid w]$')
ax.set_title('Latent Regression Structure')
ax.axhline(0, color='k', lw=0.5)
ax.legend(); ax.grid(True, alpha=0.3)

# Right: PCM item response curves
ax = axes[1]
theta_rng = np.linspace(-4, 4, 300)
item_idx, colors = 9, ['#1f77b4', '#ff7f0e', '#2ca02c']
for cat in range(m + 1):
    ax.plot(theta_rng, [pcm_probs(th, Beta[item_idx])[cat] for th in theta_rng],
            color=colors[cat], lw=2, label=f'Category {cat}')
ax.set_xlabel('Person ability ($\\theta$)')
ax.set_ylabel('Response probability')
ax.set_title(f'PCM Item {item_idx+1}: '
             f'$\\beta=({Beta[item_idx,0]:.2f}, {Beta[item_idx,1]:.2f})$')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

### 1.3 Covariate Scaling

Before passing covariates to the Stan/PyMC model, they are centered and scaled (`W_adj`)  
so that the prior $\boldsymbol{\lambda} \sim t_3(0,1)$ is weakly informative on the adjusted scale,  
regardless of the original units. After MCMC, $\hat{\boldsymbol{\lambda}}_{\text{adj}}$ is back-transformed.

In [ ]:
adj   = obtain_adjustments(W)       # shape (2, K)
W_adj = (W - adj[0]) / adj[1]      # shape (J, K)

print('adj[0] shifts:', adj[0].round(4))
print('adj[1] scales:', adj[1].round(4))
print('W_adj means (should be ~0):', W_adj.mean(0).round(4))

In [ ]:
PCM_STAN = """\
functions {
  real pcm_loglik(int y, real theta, vector beta) {
    int m = rows(beta);
    vector[m + 1] log_num;
    log_num[1] = 0.0;
    for (k in 1:m)
      log_num[k + 1] = log_num[k] + (theta - beta[k]);
    return categorical_logit_lpmf(y + 1 | log_num);
  }
  matrix get_adjustments(matrix W) {
    int J = rows(W); int K = cols(W);
    matrix[2, K] adj;
    adj[1, 1] = 0.0; adj[2, 1] = 1.0;
    for (k in 2:K) {
      real w_min = min(col(W, k)); real w_max = max(col(W, k));
      int n_extremes = 0;
      for (j in 1:J)
        n_extremes += (W[j, k] == w_min || W[j, k] == w_max);
      adj[1, k] = mean(col(W, k));
      adj[2, k] = (n_extremes == J) ? (w_max - w_min) : (sd(col(W, k)) * 2);
    }
    return adj;
  }
}
data {
  int<lower=1> I; int<lower=1> J; int<lower=1> N; int<lower=1> K;
  array[N] int<lower=1,upper=I> ii;
  array[N] int<lower=1,upper=J> jj;
  array[N] int<lower=0> y;
  matrix[J, K] W;
}
transformed data {
  array[I] int steps = rep_array(0, I);
  for (n in 1:N) if (y[n] > steps[ii[n]]) steps[ii[n]] = y[n];
  array[I] int beta_start;
  beta_start[1] = 1;
  for (i in 2:I) beta_start[i] = beta_start[i-1] + steps[i-1];
  int total_steps = sum(steps);
  matrix[2, K] adj = get_adjustments(W);
  matrix[J, K] W_adj;
  for (k in 1:K) W_adj[, k] = (W[, k] - adj[1, k]) / adj[2, k];
}
parameters {
  vector[total_steps - 1] beta_free;
  vector[J] theta;
  real<lower=0> sigma;
  vector[K] lambda_adj;
}
transformed parameters {
  vector[total_steps] beta;
  beta[1:(total_steps - 1)] = beta_free;
  beta[total_steps] = -sum(beta_free);
}
model {
  target += normal_lpdf(beta | 0, 3);
  lambda_adj ~ student_t(3, 0, 1);
  sigma ~ exponential(0.1);
  theta ~ normal(W_adj * lambda_adj, sigma);
  for (n in 1:N) {
    int i = ii[n];
    target += pcm_loglik(y[n], theta[jj[n]], segment(beta, beta_start[i], steps[i]));
  }
}
generated quantities {
  vector[K] lambda;
  for (k in 2:K) lambda[k] = lambda_adj[k] / adj[2, k];
  lambda[1] = lambda_adj[1] - dot_product(adj[1, 2:K], lambda[2:K]);
}
"""

GPCM_STAN = """\
functions {
  real gpcm_loglik(int y, real theta, real alpha, vector beta) {
    int m = rows(beta);
    vector[m + 1] log_num;
    log_num[1] = 0.0;
    for (k in 1:m)
      log_num[k + 1] = log_num[k] + (alpha * theta - beta[k]);
    return categorical_logit_lpmf(y + 1 | log_num);
  }
  matrix get_adjustments(matrix W) {
    int J = rows(W); int K = cols(W);
    matrix[2, K] adj;
    adj[1, 1] = 0.0; adj[2, 1] = 1.0;
    for (k in 2:K) {
      real w_min = min(col(W, k)); real w_max = max(col(W, k));
      int n_extremes = 0;
      for (j in 1:J)
        n_extremes += (W[j, k] == w_min || W[j, k] == w_max);
      adj[1, k] = mean(col(W, k));
      adj[2, k] = (n_extremes == J) ? (w_max - w_min) : (sd(col(W, k)) * 2);
    }
    return adj;
  }
}
data {
  int<lower=1> I; int<lower=1> J; int<lower=1> N; int<lower=1> K;
  array[N] int<lower=1,upper=I> ii;
  array[N] int<lower=1,upper=J> jj;
  array[N] int<lower=0> y;
  matrix[J, K] W;
}
transformed data {
  array[I] int steps = rep_array(0, I);
  for (n in 1:N) if (y[n] > steps[ii[n]]) steps[ii[n]] = y[n];
  array[I] int beta_start;
  beta_start[1] = 1;
  for (i in 2:I) beta_start[i] = beta_start[i-1] + steps[i-1];
  int total_steps = sum(steps);
  matrix[2, K] adj = get_adjustments(W);
  matrix[J, K] W_adj;
  for (k in 1:K) W_adj[, k] = (W[, k] - adj[1, k]) / adj[2, k];
}
parameters {
  vector<lower=0>[I] alpha;
  vector[total_steps - 1] beta_free;
  vector[J] theta;
  vector[K] lambda_adj;
}
transformed parameters {
  vector[total_steps] beta;
  beta[1:(total_steps - 1)] = beta_free;
  beta[total_steps] = -sum(beta_free);
}
model {
  alpha ~ lognormal(1, 1);
  target += normal_lpdf(beta | 0, 3);
  lambda_adj ~ student_t(3, 0, 1);
  theta ~ normal(W_adj * lambda_adj, 1.0);
  for (n in 1:N) {
    int i = ii[n];
    target += gpcm_loglik(y[n], theta[jj[n]], alpha[i], segment(beta, beta_start[i], steps[i]));
  }
}
generated quantities {
  vector[K] lambda;
  for (k in 2:K) lambda[k] = lambda_adj[k] / adj[2, k];
  lambda[1] = lambda_adj[1] - dot_product(adj[1, 2:K], lambda[2:K]);
}
"""

pcm_stan_path  = 'pcm_latent_reg.stan'
gpcm_stan_path = 'gpcm_latent_reg.stan'

with open(pcm_stan_path,  'w') as f: f.write(PCM_STAN)
with open(gpcm_stan_path, 'w') as f: f.write(GPCM_STAN)

print(f'Written: {pcm_stan_path}')
print(f'Written: {gpcm_stan_path}')

### 1.4 PCM Model — Compile with cmdstanpy

Compile `pcm_latent_reg.stan` into a CmdStan executable.  
CmdStan compiles once and caches the binary; subsequent runs skip recompilation.

In [ ]:
total_steps = I * m   # 40 total step difficulties (all items have m=2 steps)

pcm_model = CmdStanModel(stan_file=pcm_stan_path)
print(pcm_model)

### 1.5 MCMC

4 chains × (500 warmup + 500 sampling) — comparable to Furr (2017)'s 4 chains × 1000 iterations.

> **Note:** With J=500 person ability parameters, sampling takes 5–20 minutes.  
> Reduce `iter_sampling` or `J` for a quick test.

In [ ]:
stan_data_pcm = {
    'I': I, 'J': J, 'N': N, 'K': K,
    'ii': (ii + 1).tolist(),   # 0-indexed → 1-indexed for Stan
    'jj': (jj + 1).tolist(),
    'y':  pcm_y.tolist(),
    'W':  W.tolist(),
}

pcm_fit = pcm_model.sample(
    data=stan_data_pcm, chains=4,
    iter_warmup=500, iter_sampling=500,
    adapt_delta=0.9, seed=SEED, show_progress=True,
)

In [ ]:
summary = pcm_fit.summary()
key_pars = summary[summary.index.str.startswith(('sigma', 'lambda['))]
print('Posterior summary (key parameters):')
print(key_pars[['Mean', 'StdDev', '5%', '95%', 'R_hat']].to_string())
print(f'\nMax R_hat (all params): {summary["R_hat"].max():.4f}  (< 1.1 = converged)')
print()
print(pcm_fit.diagnose())

### 1.6 Parameter Recovery

**Discrepancy** = posterior mean − true generating value.  
Horizontal lines = 95% posterior interval for the discrepancy.  
Nearly all intervals should contain zero if the model recovers parameters correctly.

In [ ]:
# Extract posterior samples and reconstruct full beta vector
bf_samp   = pcm_trace.posterior['beta_free'].values.reshape(-1, total_steps - 1)
beta_samp = np.concatenate([bf_samp, -bf_samp.sum(axis=1, keepdims=True)], axis=1)
lam_adj_s = pcm_trace.posterior['lambda_adj'].values.reshape(-1, K)
sigma_s   = pcm_trace.posterior['sigma'].values.flatten()
lam_s     = back_transform_lambda(lam_adj_s, adj)   # back to original scale

# True values (row-major flatten matches PyMC's pt.reshape order)
beta_true = Beta.flatten()

rows = []
for s in range(total_steps):
    d = beta_samp[:, s] - beta_true[s]
    rows.append({'param': f'beta[{s+1}]',
                 'mean': d.mean(), 'p025': np.percentile(d, 2.5), 'p975': np.percentile(d, 97.5)})
for k in range(K):
    d = lam_s[:, k] - lambda_true[k]
    rows.append({'param': f'lambda[{k+1}]',
                 'mean': d.mean(), 'p025': np.percentile(d, 2.5), 'p975': np.percentile(d, 97.5)})
d = sigma_s - sigma_true
rows.append({'param': 'sigma', 'mean': d.mean(),
             'p025': np.percentile(d, 2.5), 'p975': np.percentile(d, 97.5)})

df_pcm = pd.DataFrame(rows)
covered = ((df_pcm['p025'] <= 0) & (df_pcm['p975'] >= 0)).sum()
print(f'95% CI covering zero: {covered}/{len(df_pcm)} = {covered/len(df_pcm)*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 14))
ax.set_facecolor('#f0f0f0')
ax.axvline(0, color='white', lw=1.5)
for i, row in df_pcm.iterrows():
    ax.plot([row['p025'], row['p975']], [i, i], 'k-', lw=1, alpha=0.7)
    ax.plot(row['mean'], i, 'ko', ms=3)
ax.set_yticks(range(len(df_pcm)))
ax.set_yticklabels(df_pcm['param'], fontsize=8)
ax.set_xlabel('Discrepancy  (posterior mean − true value)')
ax.set_title('PCM Parameter Recovery')
ax.set_xlim(-1, 1)
plt.tight_layout(); plt.show()

---
## Section 2: GPCM Simulation and Parameter Recovery

### 2.1 Model

$$
\Pr(Y_{ij} = y \mid \theta_j, \alpha_i, \boldsymbol{\beta}_i) =
\frac{\exp\sum_{s=1}^{y}(\alpha_i\theta_j - \beta_{is})}
     {1 + \sum_{k=1}^{m_i}\exp\sum_{s=1}^{k}(\alpha_i\theta_j - \beta_{is})}
\qquad
\theta_j \sim \mathcal{N}(\boldsymbol{w}_j'\boldsymbol{\lambda},\; 1)
$$

**Identification:** $\text{Var}(\theta) = 1$ is fixed; $\sigma$ is not estimated.  
**Additional prior:** $\alpha_i \sim \log\mathcal{N}(1, 1)$ (ensures $\alpha_i > 0$)

In [ ]:
np.random.seed(SEED)

# Alternating discriminations (matching Furr 2017 Section 2.3)
alpha_true = np.tile([0.8, 1.2], I // 2).astype(float)
print(f'alpha_true: {alpha_true}')

# Person abilities (sigma fixed to 1 for GPCM)
gpcm_theta = W @ lambda_true + np.random.normal(0, 1.0, J)

# Simulate GPCM responses
gpcm_y = np.array([
    simulate_response(gpcm_probs(gpcm_theta[jj[n]], alpha_true[ii[n]], Beta[ii[n]]))
    for n in range(N)
], dtype=int)

print('\nGPCM response frequency:')
for cat in range(m + 1):
    print(f'  Category {cat}: {(gpcm_y == cat).mean()*100:.1f}%')

In [ ]:
# PCM vs GPCM item response curves (same beta, different alpha)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
theta_rng = np.linspace(-4, 4, 300)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for row_idx, item_idx in enumerate([0, 9]):
    for col_idx, (label, fn, kw) in enumerate([
        ('PCM ($\\alpha=1$)',  pcm_probs,  {}),
        (f'GPCM ($\\alpha={alpha_true[item_idx]}$)', gpcm_probs,
         {'alpha': alpha_true[item_idx]})
    ]):
        ax = axes[row_idx, col_idx]
        for cat in range(m + 1):
            if 'alpha' in kw:
                p = [fn(th, kw['alpha'], Beta[item_idx])[cat] for th in theta_rng]
            else:
                p = [fn(th, Beta[item_idx])[cat] for th in theta_rng]
            ax.plot(theta_rng, p, color=colors[cat], lw=2, label=f'Cat {cat}')
        ax.set_title(f'{label}  Item {item_idx+1}\n'
                     f'$\\beta=({Beta[item_idx,0]:.2f},{Beta[item_idx,1]:.2f})$')
        ax.set_xlabel('$\\theta$'); ax.set_ylabel('Probability')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('PCM vs GPCM: Effect of Discrimination Parameter', y=1.01)
plt.tight_layout(); plt.show()

### 2.2 GPCM PyMC Model

Only three changes from the PCM model:
1. Add `alpha ~ LogNormal(1, 1)` per item
2. Replace `theta_obs - beta_obs` with `alpha_obs * theta_obs - beta_obs`
3. Fix `sigma=1.0` (not estimated)

In [ ]:
with pm.Model() as gpcm_model:

    # --- Priors ---
    alpha_rv    = pm.LogNormal('alpha', mu=1, sigma=1, shape=I)
    beta_free_g = pm.Normal('beta_free', mu=0, sigma=3, shape=total_steps - 1)
    last_g      = pt.atleast_1d(-pt.sum(beta_free_g))
    beta_g      = pm.Deterministic('beta',
                      pt.reshape(pt.concatenate([beta_free_g, last_g]), (I, m)))
    lambda_adj_g = pm.StudentT('lambda_adj', nu=3, mu=0, sigma=1, shape=K)

    # Person abilities (variance fixed to 1.0 for identification)
    theta_g = pm.Normal('theta', mu=pt.dot(W_adj_const, lambda_adj_g),
                         sigma=1.0, shape=J)

    # --- Likelihood: categorical_logit equivalent ---
    # Stan: categorical_logit_lpmf(y+1 | log_num)  where log_num uses alpha*theta
    # PyMC: pm.Categorical(p = softmax(log_num), observed = y)
    theta_obs_g = theta_g[jj]                         # shape (N,)
    alpha_obs   = alpha_rv[ii]                        # shape (N,)
    beta_obs_g  = beta_g[ii]                          # shape (N, m)

    # GPCM: alpha scales theta before subtracting step difficulties
    step_diffs_g = alpha_obs[:, None] * theta_obs_g[:, None] - beta_obs_g  # (N, m)
    unsummed_g   = pt.concatenate([pt.zeros((N, 1)), step_diffs_g], axis=1) # (N, m+1)
    log_num_g    = pt.cumsum(unsummed_g, axis=1)                             # (N, m+1)

    probs_g = pt.nnet.softmax(log_num_g)               # shape (N, m+1)
    pm.Categorical('y_obs', p=probs_g, observed=gpcm_y)

print(f'GPCM model: {gpcm_model.ndim} free parameters')

### 2.3 MCMC

In [ ]:
with gpcm_model:
    gpcm_trace = pm.sample(
        draws=500, tune=500, chains=4,
        target_accept=0.9, random_seed=SEED,
        return_inferencedata=True, progressbar=True,
    )

In [ ]:
rhat_g = az.summary(
    gpcm_trace, var_names=['alpha', 'lambda_adj'], stat_funcs=None
)[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat']]
print('GPCM posterior summary:')
print(rhat_g.to_string())

rhat_g_all = az.rhat(gpcm_trace)
max_rhat_g = max(float(v.values.max()) for v in rhat_g_all.data_vars.values())
print(f'\nMax R-hat: {max_rhat_g:.4f}')

### 2.4 GPCM Parameter Recovery

In [ ]:
alpha_s   = gpcm_trace.posterior['alpha'].values.reshape(-1, I)
bfg_s     = gpcm_trace.posterior['beta_free'].values.reshape(-1, total_steps - 1)
beta_g_s  = np.concatenate([bfg_s, -bfg_s.sum(axis=1, keepdims=True)], axis=1)
lam_adj_g = gpcm_trace.posterior['lambda_adj'].values.reshape(-1, K)
lam_g_s   = back_transform_lambda(lam_adj_g, adj)

rows_g = []
for i_item in range(I):
    d = alpha_s[:, i_item] - alpha_true[i_item]
    rows_g.append({'param': f'alpha[{i_item+1}]',
                   'mean': d.mean(), 'p025': np.percentile(d,2.5), 'p975': np.percentile(d,97.5)})
for s in range(total_steps):
    d = beta_g_s[:, s] - beta_true[s]
    rows_g.append({'param': f'beta[{s+1}]',
                   'mean': d.mean(), 'p025': np.percentile(d,2.5), 'p975': np.percentile(d,97.5)})
for k in range(K):
    d = lam_g_s[:, k] - lambda_true[k]
    rows_g.append({'param': f'lambda[{k+1}]',
                   'mean': d.mean(), 'p025': np.percentile(d,2.5), 'p975': np.percentile(d,97.5)})

df_gpcm = pd.DataFrame(rows_g)
cov_g   = ((df_gpcm['p025'] <= 0) & (df_gpcm['p975'] >= 0)).sum()
print(f'95% CI covering zero: {cov_g}/{len(df_gpcm)} = {cov_g/len(df_gpcm)*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 18))
ax.set_facecolor('#f0f0f0')
ax.axvline(0, color='white', lw=1.5)
for i, row in df_gpcm.iterrows():
    ax.plot([row['p025'], row['p975']], [i, i], 'k-', lw=1, alpha=0.7)
    ax.plot(row['mean'], i, 'ko', ms=3)
ax.set_yticks(range(len(df_gpcm)))
ax.set_yticklabels(df_gpcm['param'], fontsize=8)
ax.set_xlabel('Discrepancy  (posterior mean − true value)')
ax.set_title('GPCM Parameter Recovery')
ax.set_xlim(-1, 1)
plt.tight_layout(); plt.show()

---
## Summary

### Model Comparison

| Aspect | PCM | GPCM |
|---|---|---|
| Discrimination | Fixed: $\alpha_i = 1$ | Estimated: $\alpha_i \sim \log\mathcal{N}(1,1)$ |
| Ability SD | Estimated: $\sigma \sim \text{Exp}(0.1)$ | Fixed: $\text{Var}(\theta)=1$ |
| Latent regression | $\theta_j \sim \mathcal{N}(\mathbf{w}_j'\boldsymbol{\lambda}, \sigma^2)$ | $\theta_j \sim \mathcal{N}(\mathbf{w}_j'\boldsymbol{\lambda}, 1)$ |
| Stan source | `pcm_latent_reg.stan` (edstan) | `gpcm_latent_reg.stan` (edstan) |

### Key Mathematical Equivalence

The three implementations below are numerically identical:

| Source | Code |
|---|---|
| Original Stan | `softmax(cumulative_sum(unsummed))[y+1]` |
| Readable Stan | `log_num[y+1] - log_sum_exp(log_num)` |
| PyMC | `cumsum[n, y] - logsumexp(cumsum, axis=1)[n]` |

## References

- Furr, D. C. (2017). Partial credit and generalized partial credit models with latent regression. *Stan case study*.
- Masters, G. N. (1982). A Rasch model for partial credit scoring. *Psychometrika*, 47(2), 149–174.
- Muraki, E. (1992). A generalized partial credit model: Application of an EM algorithm. *ETS Research Report Series*, 1992(1), 1–30.
- Gelman et al. (2008). A weakly informative default prior for logistic and other regression models. *Annals of Applied Statistics*, 1360–1383.